# Prophet 预测奢侈品门店间歇性 SKU

只使用 `data/teaching/luxury_scenario_data.csv`（方案 4）与 Prophet。该 SKU 营业日约 90% 以上为零销量，正需求为 1、2、5、10 件，属于间歇性高价值需求；Prophet 假设连续、近似正态的目标，本 Notebook 展示其能力与局限。

流程：读取 CSV → 按截止日划分 → 屏蔽闭店日标签 → 拟合 Prophet（外生变量 + 公共假日）→ 预测最后 28 天 → 闭店归零 → 与基线对比。

`conditional_mean`、`occurrence_probability`、`positive_demand`、`latent_demand` 是仿真诊断真值，不作为输入特征。

In [ ]:
# 首次使用请在仓库根目录运行：uv sync --group dev
import logging
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from prophet import Prophet

logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("prophet").setLevel(logging.WARNING)


def find_repo_root():
    configured = os.environ.get("RETAIL_FORECAST_ROOT")
    candidates = (
        [Path(configured).expanduser()]
        if configured
        else [Path.cwd(), *Path.cwd().parents]
    )
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "notebooks"
        ).is_dir():
            return candidate
    raise RuntimeError(
        "找不到仓库根目录；请从仓库内启动 Jupyter，或设置 RETAIL_FORECAST_ROOT"
    )


DATA_DIR = find_repo_root() / "data" / "teaching"
HORIZON_DAYS = 28
REGRESSORS = [
    "is_saturday",
    "is_sunday",
    "is_december_peak",
    "is_vip_event",
    "discount",
]

data = pd.read_csv(DATA_DIR / "luxury_scenario_data.csv", parse_dates=["ds"])
holidays = pd.read_csv(DATA_DIR / "vic_holidays.csv", parse_dates=["ds"])
for col in REGRESSORS:
    data[col] = data[col].astype(float)

cutoff = data["ds"].iloc[-HORIZON_DAYS - 1]
train = data.loc[data.ds <= cutoff].copy()
test = data.loc[data.ds > cutoff].copy()
# 闭店日的 0 不是需求信号，训练时屏蔽。
train.loc[~train.is_open, "y"] = np.nan
assert len(test) == HORIZON_DAYS

opened = train.loc[train.is_open]
print(
    {
        "train_rows": len(train),
        "test_rows": len(test),
        "cutoff": str(cutoff.date()),
        "open_zero_rate": round(float(opened.y.eq(0).mean()), 3),
        "open_mean_sales": round(float(opened.y.mean()), 3),
    }
)

## 1. 拟合 Prophet

- 星期效应由 `is_saturday`、`is_sunday` 表达，因此关闭内置周季节性，避免重复。
- 12 月旺季由 `is_december_peak` 表达，关闭年季节性；VIP 活动与折扣作为外生变量。
- 趋势保持平缓（`changepoint_prior_scale` 小）；公共假日使用 `vic_holidays.csv`。
- 目标为稀疏计数，所有设置均为教学假设，未做调参。

In [ ]:
model = Prophet(
    growth="linear",
    weekly_seasonality=False,
    yearly_seasonality=False,
    daily_seasonality=False,
    holidays=holidays,
    changepoint_prior_scale=0.01,
    seasonality_mode="additive",
    interval_width=0.8,
)
for col in REGRESSORS:
    model.add_regressor(col, standardize=False)
model.fit(train[["ds", "y", *REGRESSORS]])

future = test[["ds", *REGRESSORS]]
forecast = model.predict(future)
forecast["is_open"] = test["is_open"].to_numpy()
# 闭店日销量为 0；开店日预测截断到非负。
forecast["y_pred"] = np.where(forecast["is_open"], np.maximum(forecast["yhat"], 0), 0)
forecast["y_lower"] = np.where(
    forecast["is_open"], np.maximum(forecast["yhat_lower"], 0), 0
)
forecast["y_upper"] = np.where(
    forecast["is_open"], np.maximum(forecast["yhat_upper"], 0), 0
)
forecast["y_true"] = test["y"].to_numpy()
forecast[["ds", "y_true", "y_pred", "y_lower", "y_upper"]].head(10)

## 2. 评估

基线：训练期营业日平均销量（营业日），闭店日为 0。间歇性需求下 RMSE 与 MAE 偏向预测均值，请同时看总量偏差。

In [ ]:
baseline = np.where(test["is_open"], opened.y.mean(), 0)


def metrics(actual, predicted):
    error = np.asarray(predicted, float) - np.asarray(actual, float)
    return {
        "MAE": np.abs(error).mean(),
        "RMSE": np.sqrt((error**2).mean()),
        "bias": error.mean(),
        "actual_total": float(np.sum(actual)),
        "predicted_total": float(np.sum(predicted)),
    }


results = pd.DataFrame(
    {
        "Prophet": metrics(forecast["y_true"], forecast["y_pred"]),
        "训练期均值基线": metrics(forecast["y_true"], baseline),
    }
).T
interval_coverage = (
    (forecast.y_true >= forecast.y_lower) & (forecast.y_true <= forecast.y_upper)
).mean()
display(results.round(3))
print(f"80% 区间覆盖率（含闭店日）：{interval_coverage:.2%}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
history = train.loc[train.ds > cutoff - pd.Timedelta(days=56)]
ax.step(history["ds"], history["y"], where="mid", color="0.5", label="train sales")
ax.step(
    forecast["ds"], forecast["y_true"], where="mid", color="#2878B5", label="test sales"
)
ax.plot(
    forecast["ds"],
    forecast["y_pred"],
    color="#D9534F",
    marker="o",
    label="Prophet forecast",
)
ax.fill_between(
    forecast["ds"],
    forecast["y_lower"],
    forecast["y_upper"],
    color="#D9534F",
    alpha=0.15,
    label="80% interval",
)
ax.axvline(cutoff, color="0.3", linestyle="--")
ax.set_title("Luxury handbag: Prophet 28-day forecast")
ax.set_ylabel("sales (units)")
ax.legend(loc="upper left", ncol=3)
ax.grid(True, alpha=0.3)
plt.show()

## 3. 解读

- Prophet 输出的是条件均值，不是 0/1/2/5/10 的离散销量；预测曲线应与总量而非单日峰值对照。
- 区间下界被截断为 0，且高斯区间无法表达稀疏尖峰，覆盖率仅作参考。
- 该 SKU 更适合 Croston/TSB 或发生概率 × 件数的两阶段模型；本 Notebook 仅作 Prophet 基准。

## 4. 间歇性需求模型：Croston、TSB、两阶段

只在营业日序列上建模，闭店日预测为 0。

- **Croston-SBA**：分别对正需求量与需求间隔做指数平滑，预测 $\hat y = (1-\alpha/2)\,\hat z/\hat p$（SBA 偏差修正）。
- **TSB**：每个营业日更新需求发生概率 $\hat p$，仅在发生时更新件数 $\hat z$，预测 $\hat p\hat z$；可随长期无需求而衰减。
- **两阶段**：逻辑回归预测发生概率（用 Prophet 相同的外生变量），乘以训练期正需求平均件数。

Croston 与 TSB 不使用外生变量，预测为常数；两阶段能利用周末、12 月、VIP 与折扣。平滑系数为教学假设，未调参。

In [ ]:
from sklearn.linear_model import LogisticRegression

ALPHA = 0.1  # 件数/间隔平滑系数
BETA = 0.05  # TSB 发生概率平滑系数


def open_sales(frame):
    return frame.loc[frame.is_open, "y"].to_numpy(float)


def croston_sba_level(y, alpha=ALPHA):
    positive = np.flatnonzero(y > 0)
    if len(positive) == 0:
        return 0.0
    size, interval = y[positive[0]], float(positive[0] + 1)
    gap = 1
    for value in y[positive[0] + 1 :]:
        if value > 0:
            size += alpha * (value - size)
            interval += alpha * (gap - interval)
            gap = 1
        else:
            gap += 1
    return (1 - alpha / 2) * size / interval


def tsb_level(y, alpha=ALPHA, beta=BETA):
    positive = y[y > 0]
    if len(positive) == 0:
        return 0.0
    prob, size = (y > 0).mean(), positive.mean()
    for value in y:
        prob += beta * (float(value > 0) - prob)
        if value > 0:
            size += alpha * (value - size)
    return prob * size


def two_stage_forecast(train_frame, test_frame):
    fit = train_frame.loc[train_frame.is_open]
    occurrence = LogisticRegression(C=10.0, max_iter=1000)
    occurrence.fit(fit[REGRESSORS], fit.y > 0)
    mean_size = fit.loc[fit.y > 0, "y"].mean()
    return occurrence.predict_proba(test_frame[REGRESSORS])[:, 1] * mean_size


def prophet_forecast(train_frame, test_frame):
    m = Prophet(
        weekly_seasonality=False,
        yearly_seasonality=False,
        daily_seasonality=False,
        holidays=holidays,
        changepoint_prior_scale=0.01,
        seasonality_mode="additive",
    )
    for col in REGRESSORS:
        m.add_regressor(col, standardize=False)
    m.fit(train_frame[["ds", "y", *REGRESSORS]])
    return np.maximum(m.predict(test_frame[["ds", *REGRESSORS]])["yhat"].to_numpy(), 0)


def all_forecasts(train_frame, test_frame):
    history = open_sales(train_frame)
    raw = {
        "Prophet": prophet_forecast(train_frame, test_frame),
        "Croston-SBA": np.full(len(test_frame), croston_sba_level(history)),
        "TSB": np.full(len(test_frame), tsb_level(history)),
        "Two-stage": two_stage_forecast(train_frame, test_frame),
    }
    is_open = test_frame["is_open"].to_numpy(bool)
    return {name: np.where(is_open, value, 0) for name, value in raw.items()}


# 最终 28 天测试
final_forecasts = all_forecasts(train, test)
final_table = pd.DataFrame(
    {name: metrics(test["y"], pred) for name, pred in final_forecasts.items()}
).T
final_table.loc["Train-mean baseline"] = metrics(test["y"], baseline)
display(final_table.round(3))

### 最终测试窗口的结果

测试窗口仅 28 天、合计约 5 件，单窗口排名几乎不可信。下一节使用滚动起点（最近 52 周，每 28 天一个起点）汇总，并用 RMSE 之外的指标衡量总量偏差。

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
styles = {
    "Prophet": "#D9534F",
    "Croston-SBA": "#7B5EA7",
    "TSB": "#2CA02C",
    "Two-stage": "#E69F00",
}
axes[0].step(
    test["ds"], test["y"], where="mid", color="#2878B5", linewidth=2, label="actual"
)
for name, pred in final_forecasts.items():
    axes[0].plot(
        test["ds"], pred, color=styles[name], marker="o", markersize=4, label=name
    )
axes[0].set_title("Final 28-day test: daily forecast")
axes[0].set_ylabel("sales (units)")
axes[0].legend(ncol=5, loc="upper left")
axes[0].grid(True, alpha=0.3)

axes[1].step(
    test["ds"],
    test["y"].cumsum(),
    where="mid",
    color="#2878B5",
    linewidth=2,
    label="actual",
)
for name, pred in final_forecasts.items():
    axes[1].plot(test["ds"], np.cumsum(pred), color=styles[name], label=name)
axes[1].set_title("Cumulative units")
axes[1].set_ylabel("cumulative sales (units)")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 滚动起点评估

13 个起点覆盖最近 364 天，每次用截止日前的全部数据重新拟合并预测下一个 28 天。`window_total_MAE` 是 28 天总量误差的平均绝对值（备货关心的量），`total_ratio` 是预测总量/实际总量（1 为无偏）。

In [ ]:
N_ORIGINS = 13
rows = []
for k in range(N_ORIGINS, 0, -1):
    stop = len(data) - HORIZON_DAYS * (k - 1)
    window_test = data.iloc[stop - HORIZON_DAYS : stop]
    window_train = data.iloc[: stop - HORIZON_DAYS].copy()
    window_train.loc[~window_train.is_open, "y"] = np.nan
    forecasts = all_forecasts(window_train, window_test)
    forecasts["Train-mean baseline"] = np.where(
        window_test["is_open"], open_sales(window_train.dropna(subset=["y"])).mean(), 0
    )
    for name, pred in forecasts.items():
        rows.append(
            pd.DataFrame(
                {
                    "origin": window_train["ds"].iloc[-1],
                    "model": name,
                    "ds": window_test["ds"].to_numpy(),
                    "y_true": window_test["y"].to_numpy(),
                    "y_pred": pred,
                }
            )
        )
rolling = pd.concat(rows, ignore_index=True)

summary = []
for name, group in rolling.groupby("model"):
    row = metrics(group["y_true"], group["y_pred"])
    totals = group.groupby("origin")[["y_true", "y_pred"]].sum()
    row["window_total_MAE"] = (totals["y_pred"] - totals["y_true"]).abs().mean()
    row["total_ratio"] = row["predicted_total"] / row["actual_total"]
    summary.append({"model": name, **row})
rolling_table = pd.DataFrame(summary).set_index("model").sort_values("MAE")
display(rolling_table.round(3))

In [ ]:
order = rolling_table.index.tolist()
colors = {**styles, "Train-mean baseline": "0.5"}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for axis, column, title in zip(
    axes,
    ["MAE", "window_total_MAE", "total_ratio"],
    [
        "Daily MAE (lower is better)",
        "28-day total abs. error",
        "Predicted / actual total",
    ],
):
    axis.bar(order, rolling_table.loc[order, column], color=[colors[n] for n in order])
    axis.set_title(title)
    axis.tick_params(axis="x", rotation=30)
    axis.grid(True, axis="y", alpha=0.3)
axes[2].axhline(1.0, color="k", linestyle="--", linewidth=1)
plt.tight_layout()
plt.show()

window_totals = rolling.groupby(["origin", "model"])["y_pred"].sum().unstack()
window_totals["actual"] = (
    rolling.drop_duplicates(["origin", "ds"]).groupby("origin")["y_true"].sum()
)
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.step(
    window_totals.index,
    window_totals["actual"],
    where="mid",
    color="#2878B5",
    linewidth=2,
    label="actual",
)
for name in ["Prophet", "Croston-SBA", "TSB", "Two-stage"]:
    ax.plot(
        window_totals.index,
        window_totals[name],
        marker="o",
        color=styles[name],
        label=name,
    )
ax.set_title("28-day total units by forecast origin")
ax.set_xlabel("forecast origin")
ax.set_ylabel("units per 28 days")
ax.legend(ncol=5)
ax.grid(True, alpha=0.3)
plt.show()

## 5. 对比分析

**总体**：13 个滚动窗口里，各模型日度 MAE 为 0.133–0.142，RMSE 为 0.241–0.257，差距很小；训练期均值基线几乎与复杂模型持平。营业日零值约 94%、全年仅约 22 件，指标差异远小于抽样噪声，不应据此宣称某模型显著更优。

**Prophet**
- 日度 MAE 与 RMSE 在比较模型中最差，总量也最偏高（预测/实际约 1.68）。
- 原因是高斯目标加外生变量系数：周末、12 月、VIP 的加性效应在稀疏数据中估计不稳，并给出非零均值；最终窗口（含 12 月旺季与 VIP 周末）预测约 8.6 件对实际 5 件。
- 优点是能给出带假日与促销效应的平滑曲线，可解释；缺点是不处理离散的 0/1/2/5/10 件分布。

**Croston-SBA 与 TSB**
- 无外生变量，预测近似常数，无法响应周末、12 月或 VIP。日度 MAE 与两阶段接近，但各窗口总量误差并不更好。
- TSB 能随长期无销售而衰减，在 2025 年末预测接近 0，窗口总量误差在各模型中最大（约 1.77），响应过快、波动大；Croston-SBA 更平稳。

**两阶段**
- 总量窗口误差最小（约 1.05），日度 MAE 与 RMSE 也属最佳或并列，并且是唯一直接用到发生概率与外生变量的间歇性模型。
- 仍整体偏高估（约 1.48）。所有模型在这 364 天均高估，是因为样本期实际销量低于数据生成过程的长期期望，而非单一模型的缺陷。

**结论**
1. 对该 SKU，两阶段（发生概率 × 平均件数）是更合适的结构：结构与生成过程一致，可解释且利用营销变量。
2. Prophet 仅在需求连续、量大的场景（方案 1）更合适；这里应只作为参考基线。
3. 评估间歇性需求应以滚动起点、总量误差和偏差为主，而不是单个 28 天窗口或仅看 RMSE。
4. 如要继续，可以评估分位数/预测分布（例如服务水平下的安全库存），并用多个随机种子重复生成数据，以降低样本噪声。